# 🛡️ Toxic Comment Detection using DistilBERT

### Project Overview
Online discussion platforms frequently struggle with abusive, toxic, and hateful content that disrupts constructive dialogue and harms participants. Manually moderating hundreds of thousands of user comments is infeasible at scale. 

In this project, we construct an end-to-end, production-grade Natural Language Processing (NLP) system for **Multi-Label Toxic Comment Classification** on the **Jigsaw Toxic Comment Dataset**. We fine-tune **DistilBERT** (`distilbert-base-uncased`), a compact and fast transformer model, to simultaneously detect 6 categories of toxicity:
1. `toxic` - General overt hostility or rudeness
2. `severe_toxic` - Extremely aggressive or threatening behavior
3. `obscene` - Vulgar, explicit, or offensive language
4. `threat` - Explicit or implicit threats of physical harm
5. `insult` - Personal derogatory attacks
6. `identity_hate` - Hate speech targeting religion, race, gender, sexual orientation, or ethnicity

### Key Highlights & Best Practices
- **Multi-Label Formulation**: Uses independent Sigmoid output activations with Binary Cross-Entropy loss (`BCEWithLogitsLoss`), permitting comments to have multiple simultaneous labels or be completely clean.
- **Hardware Acceleration**: Built and optimized for Google Colab with free NVIDIA T4 GPU using mixed precision (`fp16=True`).
- **Data Analysis & Visualization**: In-depth Exploratory Data Analysis (EDA) of label distributions, label correlations, comment lengths, class imbalance ratios, and word clouds.
- **Threshold Tuning**: Because severe categories (like `threat` and `identity_hate`) suffer from extreme class imbalance (up to 330:1), we optimize per-label classification thresholds to significantly boost macro $F_1$ score over the standard 0.5 cutoff.
- **Production Readiness**: Model saving, local zip packaging for download, Google Drive mounting, and interactive single-comment inference functions.

In [ ]:
# Install required dependencies
!pip install -q transformers datasets accelerate scikit-learn kagglehub wordcloud

In [ ]:
# Import all required libraries
import os
import re
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    classification_report,
    roc_curve,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# Silence warnings and configure plot styling
warnings.filterwarnings('ignore')
%matplotlib inline

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('seaborn-darkgrid')
sns.set_theme(style='darkgrid', palette='deep')

print("✅ Libraries imported successfully!")

In [ ]:
# Check GPU availability, set device, and set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device configured: {device}")

if torch.cuda.is_available():
    print(f"GPU Model:        {torch.cuda.get_device_name(0)}")
    print(f"GPU Count:        {torch.cuda.device_count()}")
    print(f"Allocated Memory: {torch.cuda.memory_allocated(0) / (1024 ** 2):.2f} MB")
    print(f"Cached Memory:    {torch.cuda.memory_reserved(0) / (1024 ** 2):.2f} MB")
else:
    print("⚠️ Warning: CUDA is not available. Running on CPU (training may take considerably longer).")

## 📥 Dataset Loading

We download the **Jigsaw Toxic Comment Classification Challenge** dataset from Kaggle via `kagglehub`. The dataset consists of 159,571 Wikipedia talk page comments labeled by human annotators across 6 toxicity categories.

In [ ]:
# Download dataset using kagglehub
import kagglehub

print("Downloading Jigsaw dataset using kagglehub...")
path = kagglehub.dataset_download("julian3833/jigsaw-toxic-comment-classification-challenge")
print(f"✅ Dataset downloaded to: {path}")
print("Files in dataset directory:", os.listdir(path))

In [ ]:
# Load train.csv into pandas, display shape, head, info
train_csv_path = os.path.join(path, "train.csv")
df = pd.read_csv(train_csv_path)

print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nFirst 5 rows:")
display(df.head())

print("\nDataset Info:")
df.info()

In [ ]:
# Define target toxicity labels
LABEL_COLUMNS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
print(f"Target Labels ({len(LABEL_COLUMNS)}): {LABEL_COLUMNS}")

## 📊 Exploratory Data Analysis

Before modeling, we perform comprehensive Exploratory Data Analysis (EDA):
1. **Label Distribution**: Identifying positive counts and prevalence across all 6 categories.
2. **Multi-label Co-occurrence**: Measuring pairwise correlations between categories.
3. **Comment Length**: Analyzing character distribution to inform maximum token cutoff.
4. **Clean vs. Toxic Ratio**: Comparing non-toxic comments against those with at least one flag.
5. **Word Clouds**: Contrasting dominant vocabulary in toxic versus clean comments.
6. **Class Imbalance Metrics**: Quantifying positive-to-negative imbalance ratios.

In [ ]:
# Label distribution bar chart (count of 1s per label)
label_counts = df[LABEL_COLUMNS].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("viridis", len(LABEL_COLUMNS))
bars = ax.barh(label_counts.index, label_counts.values, color=colors, edgecolor='black', alpha=0.85)

# Add value annotations on bars
max_val = max(label_counts.values)
for bar in bars:
    width = bar.get_width()
    pct = (width / len(df)) * 100
    ax.text(
        width + (max_val * 0.015),
        bar.get_y() + bar.get_height() / 2,
        f"{int(width):,}  ({pct:.2f}%)",
        va='center',
        ha='left',
        fontsize=10,
        fontweight='bold',
        color='#2c3e50'
    )

ax.set_title("Distribution of Positive Labels Across Categories", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Number of Positive Comments", fontsize=12, labelpad=10)
ax.set_ylabel("Toxicity Category", fontsize=12, labelpad=10)
ax.set_xlim(0, max_val * 1.25)

plt.tight_layout()
plt.show()

In [ ]:
# Multi-label co-occurrence heatmap (correlation matrix of labels)
corr_matrix = df[LABEL_COLUMNS].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    cbar_kws={'label': 'Pearson Correlation'},
    linewidths=0.8,
    linecolor='white',
    square=True,
    ax=ax
)

ax.set_title("Multi-Label Co-occurrence Correlation Heatmap", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Comment length distribution histogram (character length) with vertical lines for mean/median
df['char_length'] = df['comment_text'].str.len()

mean_len = df['char_length'].mean()
median_len = df['char_length'].median()

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(
    df['char_length'],
    bins=100,
    color='#3498db',
    kde=True,
    stat="density",
    alpha=0.6,
    ax=ax
)

ax.axvline(mean_len, color='#e74c3c', linestyle='--', linewidth=2.5, label=f'Mean Length: {mean_len:.1f}')
ax.axvline(median_len, color='#2ecc71', linestyle='-', linewidth=2.5, label=f'Median Length: {median_len:.1f}')

ax.set_title("Comment Length Distribution (Characters)", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Character Count", fontsize=12, labelpad=10)
ax.set_ylabel("Density", fontsize=12, labelpad=10)
ax.set_xlim(0, 1500)
ax.legend(fontsize=11, frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# Pie chart showing toxic vs clean comments ratio
df['is_toxic_any'] = (df[LABEL_COLUMNS].sum(axis=1) > 0).astype(int)
ratio_counts = df['is_toxic_any'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
pie_labels = ['Clean Comments', 'Toxic Comments (≥1 category)']
pie_colors = ['#2ecc71', '#e74c3c']
explode = (0, 0.08)

wedges, texts, autotexts = ax.pie(
    ratio_counts,
    labels=pie_labels,
    autopct=lambda p: f"{p:.1f}%\n({int(p * len(df) / 100):,} comments)",
    startangle=140,
    colors=pie_colors,
    explode=explode,
    shadow=True,
    textprops={'fontsize': 11}
)

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax.set_title("Proportion of Toxic vs. Clean Comments", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Word cloud for toxic comments vs clean comments (side by side subplots)
toxic_comments_sample = df[df['is_toxic_any'] == 1]['comment_text'].dropna().sample(n=min(5000, int(df['is_toxic_any'].sum())), random_state=42)
clean_comments_sample = df[df['is_toxic_any'] == 0]['comment_text'].dropna().sample(n=5000, random_state=42)

toxic_corpus = " ".join(toxic_comments_sample)
clean_corpus = " ".join(clean_comments_sample)

custom_stopwords = set(STOPWORDS)
custom_stopwords.update(["article", "page", "wikipedia", "talk", "one", "would", "edit", "user", "please"])

wc_toxic = WordCloud(
    width=600,
    height=400,
    background_color='white',
    colormap='Reds',
    stopwords=custom_stopwords,
    max_words=100,
    random_state=42
).generate(toxic_corpus)

wc_clean = WordCloud(
    width=600,
    height=400,
    background_color='white',
    colormap='Greens',
    stopwords=custom_stopwords,
    max_words=100,
    random_state=42
).generate(clean_corpus)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(wc_toxic, interpolation='bilinear')
axes[0].set_title("Most Frequent Words in Toxic Comments", fontsize=14, fontweight='bold', pad=12)
axes[0].axis('off')

axes[1].imshow(wc_clean, interpolation='bilinear')
axes[1].set_title("Most Frequent Words in Clean Comments", fontsize=14, fontweight='bold', pad=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Print class imbalance stats - ratio of positive to negative for each label
total_samples = len(df)
imbalance_records = []

for label in LABEL_COLUMNS:
    pos_count = int(df[label].sum())
    neg_count = total_samples - pos_count
    pos_pct = (pos_count / total_samples) * 100
    ratio = neg_count / pos_count if pos_count > 0 else float('inf')
    imbalance_records.append({
        'Category': label,
        'Positive Count': f"{pos_count:,}",
        'Negative Count': f"{neg_count:,}",
        'Positive (%)': f"{pos_pct:.2f}%",
        'Imbalance Ratio (Neg:Pos)': f"{ratio:.1f}:1"
    })

imbalance_df = pd.DataFrame(imbalance_records)
print("=" * 80)
print("                       CLASS IMBALANCE STATISTICS")
print("=" * 80)
print(imbalance_df.to_string(index=False))
print("=" * 80)
print("Key Observation: Classes like 'threat' (332:1) and 'identity_hate' (112:1) are severely")
print("underrepresented, making macro-F1 and threshold tuning essential.")

## 🔧 Data Preprocessing

Data preprocessing prepares raw user text for transformer fine-tuning:
1. **Text Cleaning**: Converting to lowercase, replacing line breaks with spaces, collapsing redundant whitespace, and trimming.
2. **Train/Val Stratified Split**: Splitting 90% train and 10% validation stratified on the `toxic` label to maintain identical positive proportions.
3. **Tokenizer Loading**: Initializing `AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')`.
4. **Token Length Analysis**: Measuring sequence token lengths to establish an efficient `max_length` (256 tokens covers over 97% of all comments while minimizing GPU memory usage).
5. **PyTorch Dataset**: Encapsulating tokenization, padding, truncation, and multi-label floating-point tensors into `ToxicDataset`.

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

print("Applying text cleaning function to all comments...")
df['comment_text'] = df['comment_text'].apply(clean_text)

print("Sample cleaned comments:")
for i, sample in enumerate(df['comment_text'].head(3), 1):
    print(f"  {i}. {sample[:110]}...")

In [ ]:
# Train/val split using sklearn train_test_split (90/10, stratify on 'toxic' column, random_state=42)
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df['toxic'],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [ ]:
# Print split sizes
print("Dataset Split Sizes:")
print(f"• Total comments:      {len(df):,}")
print(f"• Training set:        {len(train_df):,} ({len(train_df) / len(df) * 100:.1f}%)")
print(f"• Validation set:      {len(val_df):,} ({len(val_df) / len(df) * 100:.1f}%)")

print("\nVerifying Stratified Class Balance Across Splits (% Positive):")
split_comp = pd.DataFrame({
    'Train (%)': (train_df[LABEL_COLUMNS].mean() * 100).round(2),
    'Val (%)': (val_df[LABEL_COLUMNS].mean() * 100).round(2),
    'Full Dataset (%)': (df[LABEL_COLUMNS].mean() * 100).round(2)
})
print(split_comp)

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert/distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loaded Tokenizer: {MODEL_NAME}")
print(f"Vocabulary Size:  {tokenizer.vocab_size:,}")

In [ ]:
# Analyze token length distribution to pick max_length (plot histogram of token counts)
sample_comments = train_df['comment_text'].sample(n=min(5000, len(train_df)), random_state=42).tolist()
token_counts = [len(tokenizer.encode(t, truncation=False)) for t in sample_comments]
token_counts = np.array(token_counts)

pct_under_128 = (token_counts <= 128).mean() * 100
pct_under_256 = (token_counts <= 256).mean() * 100
pct_under_512 = (token_counts <= 512).mean() * 100

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(token_counts, bins=60, color='#8e44ad', kde=True, ax=ax, alpha=0.6)

ax.axvline(128, color='#f39c12', linestyle='--', linewidth=2, label=f'128 tokens ({pct_under_128:.1f}% covered)')
ax.axvline(256, color='#e74c3c', linestyle='-', linewidth=2.5, label=f'256 tokens ({pct_under_256:.1f}% covered)')
ax.axvline(512, color='#2c3e50', linestyle=':', linewidth=2, label=f'512 tokens ({pct_under_512:.1f}% covered)')

ax.set_title("Token Length Distribution (DistilBERT Tokenizer Sample)", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Number of Tokens", fontsize=12, labelpad=10)
ax.set_ylabel("Frequency", fontsize=12, labelpad=10)
ax.set_xlim(0, 600)
ax.legend(fontsize=11, frameon=True)

plt.tight_layout()
plt.show()

print(f"Token length 90th percentile: {np.percentile(token_counts, 90):.0f}")
print(f"Token length 95th percentile: {np.percentile(token_counts, 95):.0f}")
print(f"Token length 98th percentile: {np.percentile(token_counts, 98):.0f}")
print(f"Comments covered with max_length=256: {pct_under_256:.2f}% (Selected for optimal throughput and context)")

In [ ]:
# Create PyTorch Dataset class
class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = self.labels[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(labels, dtype=torch.float)
        }

In [ ]:
# Create train and val datasets
train_texts = train_df['comment_text'].tolist()
train_labels = train_df[LABEL_COLUMNS].values

val_texts = val_df['comment_text'].tolist()
val_labels = val_df[LABEL_COLUMNS].values

train_dataset = ToxicDataset(train_texts, train_labels, tokenizer, max_length=256)
val_dataset = ToxicDataset(val_texts, val_labels, tokenizer, max_length=256)

print(f"✅ Train dataset ready: {len(train_dataset):,} samples")
print(f"✅ Val dataset ready:   {len(val_dataset):,} samples")

# Inspect a single item to verify tensor shapes and types
sample_item = train_dataset[0]
print("\nSample Item Verification:")
print(f"• input_ids shape:      {sample_item['input_ids'].shape} (dtype: {sample_item['input_ids'].dtype})")
print(f"• attention_mask shape: {sample_item['attention_mask'].shape}")
print(f"• labels shape:         {sample_item['labels'].shape} (dtype: {sample_item['labels'].dtype})")
print(f"• labels values:        {sample_item['labels'].tolist()}")

## 🤖 Model Architecture

We use **DistilBERT** (`distilbert-base-uncased`), a lightweight distilled transformer model with 66 million parameters. By configuring `problem_type='multi_label_classification'` and `num_labels=len(LABEL_COLUMNS)`, Hugging Face's `AutoModelForSequenceClassification` constructs:
1. DistilBERT backbone (6 transformer layers, 768 hidden dimension, 12 attention heads).
2. Pre-classifier linear layer with ReLU and dropout.
3. 6-unit output projection layer evaluated with `nn.BCEWithLogitsLoss()`.

This ensures the model trains with independent binary cross-entropy per label rather than mutually exclusive softmax.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_COLUMNS),
    problem_type='multi_label_classification'
)
model.to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 🏋️ Training

### Hyperparameters and Configuration:
- **Learning Rate**: $2 \times 10^{-5}$ with AdamW optimizer and linear decay.
- **Batch Size**: 16 per device for training, 32 for validation.
- **Epochs**: 3 epochs with per-epoch evaluation and checkpointing.
- **Precision**: 16-bit mixed precision (`fp16=True`) accelerating training by ~2.5x on T4 GPU.
- **Best Model Selection**: `load_best_model_at_end=True` targeting `macro_f1` to prevent overfitting.

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs >= 0.5).astype(int)
    
    return {
        'macro_f1': f1_score(labels, preds, average='macro', zero_division=0),
        'micro_f1': f1_score(labels, preds, average='micro', zero_division=0),
        'macro_precision': precision_score(labels, preds, average='macro', zero_division=0),
        'macro_recall': recall_score(labels, preds, average='macro', zero_division=0),
        'roc_auc': roc_auc_score(labels, probs, average='macro'),
    }

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir='./toxic_comment_model',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    fp16=True,
    logging_steps=100,
    report_to='none',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("✅ Trainer successfully initialized and ready to train!")

In [ ]:
print('🚀 Starting training...')
train_result = trainer.train()
print('✅ Training complete!')
print(f'Training time: {train_result.metrics["train_runtime"]:.1f}s')

In [ ]:
# Plot training loss curve from trainer.state.log_history
log_history = trainer.state.log_history

train_steps = []
train_losses = []
eval_steps = []
eval_losses = []
eval_macro_f1 = []

for log in log_history:
    if 'loss' in log:
        train_steps.append(log['step'])
        train_losses.append(log['loss'])
    if 'eval_loss' in log:
        eval_steps.append(log['step'])
        eval_losses.append(log['eval_loss'])
        if 'eval_macro_f1' in log:
            eval_macro_f1.append(log['eval_macro_f1'])

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(train_steps, train_losses, label='Training Loss', color='#2980b9', alpha=0.7, linewidth=1.5)
if eval_steps:
    ax1.plot(eval_steps, eval_losses, label='Validation Loss', color='#e74c3c', marker='o', linewidth=2.5)

ax1.set_xlabel('Training Step', fontsize=12, labelpad=10)
ax1.set_ylabel('Loss (BCEWithLogitsLoss)', fontsize=12, color='#2c3e50', labelpad=10)
ax1.set_title('DistilBERT Training & Validation Loss Progression', fontsize=14, fontweight='bold', pad=15)

if eval_macro_f1:
    ax2 = ax1.twinx()
    ax2.plot(eval_steps, eval_macro_f1, label='Validation Macro F1', color='#27ae60', marker='s', linestyle='--', linewidth=2)
    ax2.set_ylabel('Validation Macro F1', fontsize=12, color='#27ae60', labelpad=10)
    ax2.grid(False)
    
    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', frameon=True)
else:
    ax1.legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

## 📈 Evaluation

In this section, we rigorously evaluate the fine-tuned model on the held-out validation dataset ($N = 15,958$):
1. **Validation Predictions**: Compute raw logits and convert to calibrated probabilities using Sigmoid.
2. **Overall Aggregate Metrics**: Macro/Micro F1, Precision, Recall, and ROC-AUC.
3. **Per-Label Classification Report**: Precision, recall, and F1 across all individual categories.
4. **Per-Label ROC-AUC Table**: Threshold-independent discrimination power.
5. **ROC Curves Plot**: Visual comparison of all 6 ROC curves against random chance.
6. **Confusion Matrices Grid**: $2 \times 3$ subplot layout of confusion matrices.
7. **Per-Label F1 Bar Chart**: Visual comparison across categories at default 0.5 threshold.

In [ ]:
# Get predictions on validation set
predictions = trainer.predict(val_dataset)
logits = predictions.predictions
labels = predictions.label_ids
probs = 1 / (1 + np.exp(-logits))
preds = (probs >= 0.5).astype(int)

print(f"Validation predictions generated for {len(preds):,} comments.")

In [ ]:
# Print overall metrics (the ones from compute_metrics)
overall_metrics = compute_metrics((logits, labels))

print("=" * 60)
print("             OVERALL VALIDATION METRICS (Threshold = 0.5)")
print("=" * 60)
for k, v in overall_metrics.items():
    print(f"• {k.replace('_', ' ').title():<22}: {v:.4f}")
print("=" * 60)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(labels, preds, target_names=LABEL_COLUMNS, zero_division=0))

In [ ]:
# Per-label ROC-AUC scores printed in a nice formatted table
roc_records = []
for i, label_name in enumerate(LABEL_COLUMNS):
    auc = roc_auc_score(labels[:, i], probs[:, i])
    p = precision_score(labels[:, i], preds[:, i], zero_division=0)
    r = recall_score(labels[:, i], preds[:, i], zero_division=0)
    f1 = f1_score(labels[:, i], preds[:, i], zero_division=0)
    roc_records.append({
        'Category': label_name,
        'ROC-AUC': f"{auc:.4f}",
        'Precision': f"{p:.4f}",
        'Recall': f"{r:.4f}",
        'F1-Score': f"{f1:.4f}"
    })

roc_df = pd.DataFrame(roc_records)
print("=" * 70)
print("                   PER-LABEL METRICS SUMMARY")
print("=" * 70)
print(roc_df.to_string(index=False))
print("=" * 70)

In [ ]:
# Plot ROC curves for each label (all 6 on one plot with legend), using matplotlib with distinct colors
fig, ax = plt.subplots(figsize=(10, 8))
palette = sns.color_palette("tab10", len(LABEL_COLUMNS))

for i, (col, color) in enumerate(zip(LABEL_COLUMNS, palette)):
    fpr, tpr, _ = roc_curve(labels[:, i], probs[:, i])
    auc_val = roc_auc_score(labels[:, i], probs[:, i])
    ax.plot(fpr, tpr, color=color, linewidth=2.2, label=f'{col} (AUC = {auc_val:.4f})')

ax.plot([0, 1], [0, 1], color='#7f8c8d', linestyle='--', linewidth=1.5, label='Random Guessing (AUC = 0.5000)')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12, labelpad=10)
ax.set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12, labelpad=10)
ax.set_title('Receiver Operating Characteristic (ROC) Curves by Category', fontsize=14, fontweight='bold', pad=15)
ax.legend(loc="lower right", fontsize=11, frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for each label as a 2x3 subplot grid using seaborn heatmap with annotations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(LABEL_COLUMNS):
    cm = confusion_matrix(labels[:, i], preds[:, i])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=axes[i],
        xticklabels=['Clean (0)', 'Toxic (1)'],
        yticklabels=['Clean (0)', 'Toxic (1)']
    )
    axes[i].set_title(f"Label: {col}", fontsize=13, fontweight='bold', pad=10)
    axes[i].set_xlabel("Predicted Class", fontsize=10)
    axes[i].set_ylabel("True Class", fontsize=10)

plt.suptitle("Confusion Matrices per Toxicity Category (Threshold = 0.5)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Plot per-label F1 scores as a horizontal bar chart
f1_per_label = [f1_score(labels[:, i], preds[:, i], zero_division=0) for i in range(len(LABEL_COLUMNS))]
f1_series = pd.Series(f1_per_label, index=LABEL_COLUMNS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("mako", len(LABEL_COLUMNS))
bars = ax.barh(f1_series.index, f1_series.values, color=colors, edgecolor='black', alpha=0.85)

for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"{width:.4f}",
        va='center',
        ha='left',
        fontsize=10,
        fontweight='bold',
        color='#2c3e50'
    )

ax.set_title("Per-Label F1 Scores at Default Threshold (0.50)", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("F1 Score", fontsize=12, labelpad=10)
ax.set_ylabel("Toxicity Category", fontsize=12, labelpad=10)
ax.set_xlim(0, 1.05)

plt.tight_layout()
plt.show()

## 🎯 Threshold Tuning

The default binary classification threshold of **0.50** is rarely optimal under strong class imbalance. In rare categories such as `threat` (which makes up only 0.3% of the dataset) or `identity_hate` (~0.9%), the model's predicted probability might rarely exceed 0.5 even when there is substantial evidence of toxicity.

By systematically sweeping candidate thresholds from **0.10 to 0.90** (step 0.05) independently for each category, we locate the exact decision boundary that optimizes the trade-off between Precision and Recall to maximize the **$F_1$ score**.

In [ ]:
# Sweep thresholds 0.1 to 0.9 (step 0.05), find threshold that maximizes F1. Store in optimal_thresholds dict. Print results table.
threshold_candidates = np.arange(0.1, 0.95, 0.05)
optimal_thresholds = {}
tuning_summary = []

for i, col in enumerate(LABEL_COLUMNS):
    f1_at_05 = f1_score(labels[:, i], (probs[:, i] >= 0.5).astype(int), zero_division=0)
    best_thresh = 0.50
    best_f1 = f1_at_05
    
    for t in threshold_candidates:
        candidate_preds = (probs[:, i] >= t).astype(int)
        score = f1_score(labels[:, i], candidate_preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresh = round(float(t), 2)
            
    optimal_thresholds[col] = best_thresh
    tuning_summary.append({
        'Category': col,
        'Default Thresh': 0.50,
        'Default F1': f"{f1_at_05:.4f}",
        'Optimal Thresh': f"{best_thresh:.2f}",
        'Optimal F1': f"{best_f1:.4f}",
        'F1 Delta': f"{best_f1 - f1_at_05:+.4f}"
    })

tuning_df = pd.DataFrame(tuning_summary)

print("=" * 80)
print("                   OPTIMAL THRESHOLD CALIBRATION RESULTS")
print("=" * 80)
print(tuning_df.to_string(index=False))
print("=" * 80)
print(f"\nOptimal Thresholds Dictionary:\n{optimal_thresholds}")

In [ ]:
# Re-evaluate with optimal thresholds and print improved classification report
preds_optimal = np.zeros_like(preds)
for i, col in enumerate(LABEL_COLUMNS):
    preds_optimal[:, i] = (probs[:, i] >= optimal_thresholds[col]).astype(int)

print("=" * 70)
print("     CLASSIFICATION REPORT WITH OPTIMAL THRESHOLDS")
print("=" * 70)
print(classification_report(labels, preds_optimal, target_names=LABEL_COLUMNS, zero_division=0))
print("=" * 70)

macro_f1_default = f1_score(labels, preds, average='macro', zero_division=0)
macro_f1_optimal = f1_score(labels, preds_optimal, average='macro', zero_division=0)
print(f"\n🎯 Macro F1 Improvement: {macro_f1_default:.4f} ➔ {macro_f1_optimal:.4f} ({macro_f1_optimal - macro_f1_default:+.4f})")

In [ ]:
# Compare default vs optimized F1 scores as grouped bar chart
f1_default_scores = [f1_score(labels[:, i], preds[:, i], zero_division=0) for i in range(len(LABEL_COLUMNS))]
f1_optimal_scores = [f1_score(labels[:, i], preds_optimal[:, i], zero_division=0) for i in range(len(LABEL_COLUMNS))]

x = np.arange(len(LABEL_COLUMNS))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, f1_default_scores, width, label='Default (0.50)', color='#95a5a6', edgecolor='black', alpha=0.85)
rects2 = ax.bar(x + width/2, f1_optimal_scores, width, label='Optimized Threshold', color='#2ecc71', edgecolor='black', alpha=0.85)

ax.set_ylabel('F1 Score', fontsize=12, labelpad=10)
ax.set_title('Default vs. Threshold-Optimized F1 Scores by Category', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(LABEL_COLUMNS, fontsize=11, rotation=15)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11, frameon=True)

# Add improvement delta annotations above bars
for i in range(len(LABEL_COLUMNS)):
    diff = f1_optimal_scores[i] - f1_default_scores[i]
    if diff > 0.001:
        ax.annotate(
            f"+{diff:.3f}",
            xy=(x[i] + width/2, f1_optimal_scores[i]),
            xytext=(0, 6),
            textcoords="offset points",
            ha='center',
            fontsize=9,
            fontweight='bold',
            color='#27ae60'
        )

plt.tight_layout()
plt.show()

## 🔍 Inference Demo

We define a standalone prediction function `predict_toxicity` that takes arbitrary text strings, applies identical text cleaning, tokenizes the input, runs forward inference, applies the calibrated per-category thresholds, and outputs probabilities alongside categorical toxicity flags.

In [ ]:
def predict_toxicity(text, model, tokenizer, thresholds=None, device='cuda'):
    if thresholds is None:
        thresholds = {col: 0.5 for col in LABEL_COLUMNS}
    
    model.eval()
    encoding = tokenizer(
        clean_text(text),
        max_length=256,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**encoding)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]
    
    results = {}
    for label, prob in zip(LABEL_COLUMNS, probs):
        results[label] = {
            'probability': float(prob),
            'is_toxic': bool(prob >= thresholds[label])
        }
    return results

In [ ]:
test_comments = [
    "You are an amazing person and I appreciate your contributions!",
    "You're such an idiot, go kill yourself",
    "I disagree with your point but respect your opinion",
    "This article is garbage, the author should be ashamed",
    "I will find you and make you pay for this"
]

for comment in test_comments:
    print(f'\n💬 "{comment}"')
    result = predict_toxicity(comment, model, tokenizer, optimal_thresholds, device=device)
    for label, info in result.items():
        emoji = '🔴' if info['is_toxic'] else '🟢'
        print(f'  {emoji} {label}: {info["probability"]:.3f}')
    print('-' * 60)

## 💾 Save Model

We preserve model weights, tokenizer vocabularies, and optimal threshold mappings:
1. **Google Drive**: For persistent cloud storage across Colab runtimes.
2. **Local Directory & ZIP File**: Compressed archive ready for download or packaging into production serving containers (FastAPI, Triton, etc.).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json

save_path = '/content/drive/MyDrive/toxic_comment_model'
os.makedirs(save_path, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Save optimal thresholds
with open(f'{save_path}/optimal_thresholds.json', 'w') as f:
    json.dump(optimal_thresholds, f)

print(f'✅ Model saved to {save_path}')

In [ ]:
local_save_path = './toxic_comment_model_final'
os.makedirs(local_save_path, exist_ok=True)
model.save_pretrained(local_save_path)
tokenizer.save_pretrained(local_save_path)
with open(f'{local_save_path}/optimal_thresholds.json', 'w') as f:
    json.dump(optimal_thresholds, f)

# Zip for easy download
!zip -r toxic_comment_model_final.zip toxic_comment_model_final/
print('✅ Model zipped for download')

## 📋 Training Summary

The summary dashboard below consolidates all model specifications, dataset metrics, overall evaluation scores, and calibrated threshold benchmarks into a single clean reference card.

In [ ]:
# Print a summary dashboard with all key metrics, model info, training time, dataset stats
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 72)
print("            🛡️ TOXIC COMMENT CLASSIFICATION SUMMARY DASHBOARD        ")
print("=" * 72)
print(f"• Pretrained Model:          {MODEL_NAME}")
print(f"• Total Parameters:          {total_params:,}")
print(f"• Trainable Parameters:      {trainable_params:,}")
print(f"• Classification Task:       Multi-label (6 independent sigmoid outputs)")
print(f"• Target Categories:         {', '.join(LABEL_COLUMNS)}")
print("-" * 72)
print("DATASET & SPLIT SUMMARY:")
print(f"• Total Comments:            {len(df):,}")
print(f"• Training Samples:          {len(train_df):,} ({len(train_df) / len(df) * 100:.1f}%)")
print(f"• Validation Samples:        {len(val_df):,} ({len(val_df) / len(df) * 100:.1f}%)")
print(f"• Sequence Max Length:       256 tokens")
print("-" * 72)
print("OVERALL PERFORMANCE BENCHMARKS:")
print(f"• Default Macro F1 (t=0.50): {macro_f1_default:.4f}")
print(f"• Calibrated Macro F1:       {macro_f1_optimal:.4f} (Improvement: {macro_f1_optimal - macro_f1_default:+.4f})")
print(f"• Macro ROC-AUC:             {overall_metrics['roc_auc']:.4f}")
print(f"• Micro F1:                  {overall_metrics['micro_f1']:.4f}")
print(f"• Macro Precision:           {overall_metrics['macro_precision']:.4f}")
print(f"• Macro Recall:              {overall_metrics['macro_recall']:.4f}")
print("-" * 72)
print("PER-CATEGORY CALIBRATED THRESHOLDS & METRICS:")
for i, col in enumerate(LABEL_COLUMNS):
    thresh = optimal_thresholds[col]
    f1_val = f1_score(labels[:, i], preds_optimal[:, i], zero_division=0)
    auc_val = roc_auc_score(labels[:, i], probs[:, i])
    print(f"  • {col:<15} | Optimal Thresh: {thresh:.2f} | F1: {f1_val:.4f} | ROC-AUC: {auc_val:.4f}")
print("=" * 72)
print("🎉 Model successfully trained, evaluated, calibrated, and ready for deployment!")
print("=" * 72)